# Beta Pictoris orbit fit

The [radial velocity](https://jax.exoplanet.codes/en/latest/tutorials/rv/) and
[relative astrometry](../exoplanet_astrometry_fitting.ipynb) tutorials
demonstrate how `jaxoplanet` can be used to fit the RV and astrometric data, respectively.
In this tutorial, we will demonstrate how to model both simultaneously for the same system.
We will use observations from the $\beta$ Pictoris system to reproduce the results from [Vandal et al. (2020)](https://iopscience.iop.org/article/10.3847/1538-3881/abba30).

In [ ]:
import jax
import numpyro

# For multi-core parallelism (useful when running multiple MCMC chains in parallel)
numpyro.set_host_device_count(4)

# For CPU (use "gpu" for GPU)
numpyro.set_platform("cpu")

jax.config.update("jax_enable_x64", True)

## Relative astrometry data

We compiled the relative astrometry data in a single CSV with the `time` in MJD, the separation (`sep`) in mas, the position angle (`pa`) in degrees. The instrument is also given in an extra column.

In [ ]:
import pandas as pd
df_astro = pd.read_csv("https://raw.githubusercontent.com/vandalt/jaxoplanet-astrometry/refs/heads/main/data/astrometry.csv")
df_astro.head()

To ensure the data works well with our JAX models below, let us first extract it to Numpy arrays.
We will also derive the R.A. and Dec values for plotting purposes.

In [ ]:
import numpy as np

t_astro = df_astro.time.values
sep = df_astro.sep.values
pa = df_astro.pa.values
sep_err = df_astro.sep_err.values
pa_err = df_astro.pa_err.values

pa_rad = np.radians(pa)
ra = sep * np.sin(pa_rad)
dec = sep * np.cos(pa_rad)
pa_err_rad = np.radians(pa_err)
dec_err = np.sqrt(
    (np.cos(pa_rad)**2 * sep_err**2) +
    (sep**2 * np.sin(pa_rad)**2 * pa_err_rad**2)
)
ra_err = np.sqrt(
    (np.sin(pa_rad)**2 * sep_err**2) +
    (sep**2 * np.cos(pa_rad)**2 * pa_err_rad**2)
)
df_astro["ra"] = ra
df_astro["ra_err"] = ra_err
df_astro["dec"] = dec
df_astro["dec_err"] = dec_err

In [ ]:
import matplotlib.pyplot as plt

_, ax = plt.subplots(figsize=(4, 4))
ax.errorbar(df_astro.ra, df_astro.dec, xerr=df_astro.ra_err, yerr=df_astro.dec_err, fmt="k.")
ax.plot(0, 0, "k*", markersize=15)
ax.axis("equal")
ax.invert_xaxis()
ax.set_xlabel("$\\Delta$ RA [mas]")
ax.set_ylabel("$\\Delta$ Dec [mas]")
plt.tight_layout()
plt.show()

In [ ]:
fix, axs = plt.subplots(nrows=2, sharex=True, figsize=(6, 8))
axs[0].errorbar(df_astro.time, df_astro.sep, yerr=df_astro.sep_err, fmt="k.")
axs[0].set_ylabel("Sep [mas]")
axs[1].errorbar(df_astro.time, df_astro.pa, yerr=df_astro.pa_err, fmt="k.")
axs[1].set_ylabel("PA [deg]")
axs[1].set_ylim(0, 360)

axs[-1].set_xlabel("Time [MJD]")
plt.show()

## RV Data

We also compiled the post-processed HARPS radial velocities from Table 3 of [Vandal et al. (2020)](https://iopscience.iop.org/article/10.3847/1538-3881/abba30) in a CSV.
The time is in MJD and the RVs are in m/s.

In [ ]:
df_rv = pd.read_csv("https://raw.githubusercontent.com/vandalt/jaxoplanet-astrometry/refs/heads/main/data/rv.csv")
df_rv.head()

Again, we extract the values to Numpy arrays

In [ ]:
t_rv = df_rv.mjd.values
rv = df_rv.rv.values
rv_err = df_rv.rv_err.values

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(df_rv.mjd, df_rv.rv, yerr=df_rv.rv_err, fmt="k.")
ax.set_ylabel("RV [m/s]")
ax.set_xlabel("MJD")
plt.show()

## Forward model

We have two types of data, so we will need two separate forward models.
We could group everything under one model, but this would not be particularly efficient since the times are not the same for the two data types.

Instead, we will create a `build_system()` function that creates our `System` object for both forward models.

In [ ]:
def build_system(params):
    # Radius is required by jaxoplanet but not used in the RV or astrometry models
    system = System(
        Central(mass=params["m_tot"], radius=1.0)
    ).add_body(
        Body(
            time_transit=params["Tc"],
            period=params.get("P", None),
            semimajor=params.get("a", None),
            inclination=params["i"],
            eccentricity=params["e"],
            omega_peri=params["omega"],
            sin_asc_node=jnp.sin(params["Omega"]),
            cos_asc_node=jnp.cos(params["Omega"]),
            parallax=params["parallax"],
            mass=params.get("mass", None)
        )
    )
    return system

A few things to note here:

- Masses are in units of `M_sun`
- Angles are in radians
- The semi-major axis is in units of `R_sun`
- The parallax is in arcsec

### Astrometric model

The astrometric part of the model with compute separation and position angle at all provided times.
`jaxoplanet` returns them in arcsec and radians, so we convert to mas and degrees to be consistent with the data.

In [ ]:
from jaxoplanet.orbits.keplerian import System, Body, Central
import jax.numpy as jnp

def astrometry_model(time, params):
    system = build_system(params)
    sep, pa = system.bodies[0].relative_angles(time, parallax=params["parallax"])
    sep = sep * 1e3
    pa = pa / deg
    return sep, pa % 360

We can now test the model with manually selected parameter values.
We picked values not too far from the priors and posteriors of the paper to test the forward model and make sure it works as expected.

In [ ]:
import astropy.units as u
from jaxoplanet import constants

deg = np.pi / 180
yr = 365.25

init_params = {
    "a": 8.5 * constants.au,
    "Tc": 58010,
    "i": 89 * deg,
    "e": 0.0,
    "omega": 13 * deg,
    "Omega": 32 * deg,
    "parallax": 51.44 * 1e-3,
    "m_tot": 1.6,
    "mass": 10.0 * constants.M_jup,
}

In [ ]:
t_fine = np.linspace(t_astro.min() - 10 * yr, t_astro.max() + 10 * yr, num=1000)
sep_fine, pa_fine = astrometry_model(t_fine, init_params)
sep_mod, pa_mod = astrometry_model(t_astro, init_params)

fix, axs = plt.subplots(nrows=4, sharex=True, figsize=(6, 8), gridspec_kw={"height_ratios": (2, 1, 2, 1)})
axs[0].errorbar(df_astro.time, df_astro.sep, yerr=df_astro.sep_err, fmt="k.")
axs[0].plot(t_fine, sep_fine)
axs[0].set_ylabel("Sep [mas]")

axs[1].errorbar(df_astro.time, df_astro.sep - sep_mod, yerr=df_astro.sep_err, fmt="k.")
axs[1].set_ylabel("Sep residuals [mas]")
axs[1].axhline(0, linestyle="--")

axs[2].errorbar(df_astro.time, df_astro.pa, yerr=df_astro.pa_err, fmt="k.")
axs[2].plot(t_fine, pa_fine)
axs[2].set_ylabel("PA [deg]")

axs[3].errorbar(df_astro.time, df_astro.pa - pa_mod, yerr=df_astro.sep_err, fmt="k.")
axs[3].axhline(0, linestyle="--")
axs[3].set_ylabel("PA residuals [deg]")

axs[-1].set_xlabel("Time [MJD]")
plt.show()

In [ ]:
ra_fine = sep_fine * np.sin(pa_fine * deg)
dec_fine = sep_fine * np.cos(pa_fine * deg)
_, ax = plt.subplots(figsize=(4, 4))
ax.errorbar(df_astro.ra, df_astro.dec, xerr=df_astro.ra_err, yerr=df_astro.dec_err, fmt="k.")
ax.plot(ra_fine, dec_fine)
ax.plot(0, 0, "k*", markersize=15)
ax.axis("equal")
ax.invert_xaxis()
ax.set_xlabel("$\\Delta$ RA [mas]")
ax.set_ylabel("$\\Delta$ Dec [mas]")
plt.tight_layout()
plt.show()

Not too bad for a first guess!

### RV Model

We can now do the same for radial velocities.
`jaxoplanet`'s `System.radial_velocity()` function returns the star's RVs in m/s.
Each element in the list gives the reflex motion corresponding to one of the companions.

In [ ]:
def rv_model(time, params):
    system = build_system(params)
    return system.radial_velocity(time)[0]

In [ ]:
t_fine_rv = np.linspace(t_rv.min(), t_rv.max(), num=1000)
rv_fine = rv_model(t_fine_rv, init_params)
rv_mod = rv_model(t_rv, init_params)

fig, axs = plt.subplots(2, 1, figsize=(8, 8), gridspec_kw={"height_ratios": (2, 1)})
axs[0].errorbar(df_rv.mjd, df_rv.rv, yerr=df_rv.rv_err, fmt="k.")
axs[0].plot(t_fine_rv, rv_fine)
axs[0].set_ylabel("RV [m/s]")

axs[1].errorbar(df_rv.mjd, df_rv.rv - rv_mod, yerr=df_rv.rv_err, fmt="k.")
axs[1].axhline(0, linestyle="--")
axs[1].set_ylabel("RV residuals [m/s]")

axs[-1].set_xlabel("MJD")

plt.show()

## Probabilistic model

To sample this forward model, we will need to wrap it in a Numpyro model.
We implement the same priors as those from Table 2 of [Vandal et al. (2020)](https://iopscience.iop.org/article/10.3847/1538-3881/abba30).

However, here a are a few differences worth noting:

- We do not include $\beta$ Pic c in our model
- We do not include the two correction parameters between SPHERE and GPI
- We do not include the GRAVITY observation with correlated uncertainties

All of the above would make the model more realistic, but we will try to keep things simple for this tutorial.

In [ ]:
import numpyro
from numpyro import distributions as dist, infer
from numpyro_ext import distributions as distx

def model():
    # Time of conjunction uniform in MJD
    tc = numpyro.sample("tc", dist.Uniform(57000, 59000))

    # h=sqrt(e)*cos(omega), k=sqrt(e)*sin(omega)
    hk = numpyro.sample("hk", distx.UnitDisk())
    h, k = hk[..., 0], hk[..., 1]
    numpyro.deterministic("h", h)
    numpyro.deterministic("k", k)
    e = numpyro.deterministic("e", h**2 + k**2)
    omega = numpyro.deterministic("omega", jnp.arctan2(k, h))

    # Prior uniform in log-a between 1 and 100 AU. Convert to R_sun below
    loga = numpyro.sample("loga", dist.Uniform(jnp.log(1), jnp.log(100)))
    a = numpyro.deterministic("a", jnp.exp(loga))

    # Mass of beta pic b in M_jup, convert to M_sun below
    mass = numpyro.sample("mass", dist.Uniform(1, 20))

    Omega = numpyro.sample("Omega", dist.Uniform(18 * deg, 90 * deg))

    # equivalent to the sin(i) prior in the paper
    cosi = numpyro.sample("cosi", dist.Uniform(-1, 1))
    i = numpyro.deterministic("i", jnp.arccos(cosi))

    # Parallax in arcsec
    plx = numpyro.sample("plx", dist.Normal(51.44e-3, 0.12e-3))

    # RV offset in m/s
    gamma = numpyro.sample("gamma", dist.Uniform(-100, 100))

    # Total mass of the system in M_sun
    m_tot = numpyro.sample("m_tot", dist.Uniform(1.4, 2.0))

    # Note the conversion M_jup->M_sun for mass and AU->R_sun for a
    params = {
        "m_tot": m_tot,
        "mass": mass * constants.M_jup,
        "Tc": tc,
        "a": a * constants.au,
        "e": e,
        "i": i,
        "omega": omega,
        "Omega": Omega,
        "parallax": plx,
    }

    # Save model output at observation time for the likelihood
    rv_mod = rv_model(t_rv, params) - gamma
    sep_mod, pa_mod = astrometry_model(t_astro, params)
    sep_tot_err = sep_err
    pa_tot_err = pa_err
    rv_tot_err = rv_err

    # Likelihood on sep and pa
    # The likelihood for pa accounts for wrapping of the angle
    numpyro.sample("sep_obs", dist.Normal(loc=sep_mod, scale=sep_tot_err), obs=sep)
    numpyro.sample("rv_obs", dist.Normal(loc=rv_mod, scale=rv_tot_err), obs=rv)
    pa_diff = jnp.arctan2(
        jnp.sin(pa_mod * deg - pa * deg), jnp.cos(pa_mod * deg - pa * deg)
    )
    numpyro.sample("pa_obs", dist.Normal(loc=pa_diff, scale=pa_tot_err * deg), obs=0.0)

    # Save finer grid for plotting
    sep_dense, pa_dense = astrometry_model(t_fine, params)
    rv_dense = rv_model(t_fine_rv, params) - gamma
    sep_save = numpyro.deterministic("sep_save", sep_dense)
    pa_save = numpyro.deterministic("pa_save", pa_dense)
    rv_save = numpyro.deterministic("rv_save", rv_dense)

### Prior predictive

It is always a good idea to check the prior in parameter space and in model space before going futher with the inference.

In [ ]:
import arviz as az
import corner
n_prior_samples = 2000

key = jax.random.key(0)

key, subkey = jax.random.split(key)
prior_samples = infer.Predictive(model, num_samples=n_prior_samples)(subkey)

converted_prior_samples = {
    f"{p}": np.expand_dims(prior_samples[p], axis=0) for p in prior_samples
}
var_names = ["tc", "h", "k", "loga", "mass", "Omega", "cosi", "plx", "gamma", "m_tot"]
det_names = ["e", "omega", "a", "i"]
prior_idata = az.from_dict(converted_prior_samples)

corner.corner(prior_idata, var_names=var_names + det_names)
plt.show()

In [ ]:
rng = np.random.default_rng()

n_plots = 100
plot_inds = rng.integers(n_prior_samples, size=n_plots)

fig, axs = plt.subplots(nrows=2, sharex=True, figsize=(6, 8))
axs[0].errorbar(df_astro.time, df_astro.sep, yerr=df_astro.sep_err, fmt="k.")
axs[0].set_ylabel("Sep [mas]")
axs[1].errorbar(df_astro.time, df_astro.pa, yerr=df_astro.pa_err, fmt="k.")
axs[1].set_ylabel("PA [deg]")
for idx in plot_inds:
    axs[0].plot(t_fine, prior_idata.posterior["sep_save"].values[0, idx], "C1", alpha=0.1)
    axs[1].plot(t_fine, prior_idata.posterior["pa_save"].values[0, idx], "C1", alpha=0.1)

axs[-1].set_xlabel("Time [MJD]")
plt.show()

In [ ]:
_, ax = plt.subplots(figsize=(4, 4))
ax.errorbar(df_astro.ra, df_astro.dec, xerr=df_astro.ra_err, yerr=df_astro.dec_err, fmt="k.")
for idx in plot_inds:
    sep_fine = prior_idata.posterior["sep_save"].values[0, idx]
    pa_fine = prior_idata.posterior["pa_save"].values[0, idx]
    ra_fine = sep_fine * np.sin(pa_fine * deg)
    dec_fine = sep_fine * np.cos(pa_fine * deg)
    ax.plot(ra_fine, dec_fine, "C1", alpha=0.1)
ax.plot(0, 0, "k*", markersize=15)
ax.axis("equal")
ax.invert_xaxis()
ax.set_xlabel("$\\Delta$ RA [mas]")
ax.set_ylabel("$\\Delta$ Dec [mas]")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(df_rv.mjd, df_rv.rv, yerr=df_rv.rv_err, fmt="k.")
for idx in plot_inds:
    ax.plot(t_fine_rv, prior_idata.posterior["rv_save"].values[0, idx], "C1", alpha=0.1)
ax.set_ylabel("RV [m/s]")
ax.set_xlabel("MJD")
plt.show()

The prior covers a wide variety of solution model space, including some that are a bit too wide for the data but let's keep things consistent with the paper.

## Optimization

First we can try to optimize the model and overplot the solution the data.


In [ ]:
import numpyro_ext.optim as optimx

print(f"Optimizing the model")
#init = infer.init_to_value(values=start)
init = infer.init_to_median()
key, subkey = jax.random.split(key)
map_soln = optimx.optimize(model, init_strategy=init)(subkey)

In [ ]:
rng = np.random.default_rng()

fig, axs = plt.subplots(nrows=2, sharex=True, figsize=(6, 8))
axs[0].errorbar(df_astro.time, df_astro.sep, yerr=df_astro.sep_err, fmt="k.")
axs[0].set_ylabel("Sep [mas]")
axs[1].errorbar(df_astro.time, df_astro.pa, yerr=df_astro.pa_err, fmt="k.")
axs[1].set_ylabel("PA [deg]")
axs[0].plot(t_fine, map_soln["sep_save"], "C1")
axs[1].plot(t_fine, map_soln["pa_save"], "C1")

axs[-1].set_xlabel("Time [MJD]")
plt.show()

In [ ]:
sep_fine = map_soln["sep_save"]
pa_fine = map_soln["pa_save"]
ra_fine = sep_fine * np.sin(pa_fine * deg)
dec_fine = sep_fine * np.cos(pa_fine * deg)
_, ax = plt.subplots(figsize=(4, 4))
ax.errorbar(df_astro.ra, df_astro.dec, xerr=df_astro.ra_err, yerr=df_astro.dec_err, fmt="k.")
ax.plot(ra_fine, dec_fine, "C1")
ax.plot(0, 0, "k*", markersize=15)
ax.axis("equal")
ax.invert_xaxis()
ax.set_xlabel("$\\Delta$ RA [mas]")
ax.set_ylabel("$\\Delta$ Dec [mas]")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(df_rv.mjd, df_rv.rv, yerr=df_rv.rv_err, fmt="k.")
ax.plot(t_fine_rv, map_soln["rv_save"], "C1")
ax.set_ylabel("RV [m/s]")
ax.set_xlabel("MJD")
plt.show()

In [ ]:
import pprint
print("MAP Solution")
pprint.pprint({v: map_soln[v].item() for v in var_names})
print("MAP Derived parameters")
pprint.pprint({v: map_soln[v].item() for v in det_names})

## Sampling

Then we can sample the model to derive uncertainties on all parameters.

In [ ]:
sampler = infer.MCMC(
    infer.NUTS(
        model,
        dense_mass=True,
        regularize_mass_matrix=True,
        init_strategy=infer.init_to_value(values=map_soln)
    ),
    num_warmup=1000,
    num_samples=1000,
    num_chains=2,
    progress_bar=True,
)
key, subkey = jax.random.split(key)
sampler.run(subkey)

In [ ]:
idata = az.from_numpyro(sampler)

We can summarize the sample on the fitted parameters first.

In [ ]:
az.summary(idata, var_names=var_names)

In [ ]:
corner.corner(idata, var_names=var_names)
plt.show()

And we can also summarize the inference for some derived parameters that were not directly sampled.

In [ ]:
az.summary(idata, var_names=det_names)

In [ ]:
corner.corner(idata, var_names=det_names)
plt.show()

### Posterior predictive

And finally, we can show model predictions from our posterior samples in astrometry and in RV.

In [ ]:
n_plots = 100
n_posterior_samples = idata.posterior.chain.size * idata.posterior.draw.size
plot_inds = rng.integers(n_posterior_samples, size=n_plots)

fig, axs = plt.subplots(nrows=2, sharex=True, figsize=(6, 8))
axs[0].errorbar(df_astro.time, df_astro.sep, yerr=df_astro.sep_err, fmt="k.")
axs[0].set_ylabel("Sep [mas]")
axs[1].errorbar(df_astro.time, df_astro.pa, yerr=df_astro.pa_err, fmt="k.")
axs[1].set_ylabel("PA [deg]")
for idx in plot_inds:
    axs[0].plot(t_fine, idata.posterior["sep_save"].values.reshape(-1, t_fine.size)[idx], "C1", alpha=0.1)
    axs[1].plot(t_fine, idata.posterior["pa_save"].values.reshape(-1, t_fine.size)[idx], "C1", alpha=0.1)

axs[-1].set_xlabel("Time [MJD]")
plt.show()

In [ ]:
_, ax = plt.subplots(figsize=(4, 4))
ax.errorbar(df_astro.ra, df_astro.dec, xerr=df_astro.ra_err, yerr=df_astro.dec_err, fmt="k.")
for idx in plot_inds:
    sep_fine = idata.posterior["sep_save"].values.reshape(-1, t_fine.size)[idx]
    pa_fine = idata.posterior["pa_save"].values.reshape(-1, t_fine.size)[idx]
    ra_fine = sep_fine * np.sin(pa_fine * deg)
    dec_fine = sep_fine * np.cos(pa_fine * deg)
    ax.plot(ra_fine, dec_fine, "C1", alpha=0.1)
ax.plot(0, 0, "k*", markersize=15)
ax.axis("equal")
ax.invert_xaxis()
ax.set_xlabel("$\\Delta$ RA [mas]")
ax.set_ylabel("$\\Delta$ Dec [mas]")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(df_rv.mjd, df_rv.rv, yerr=df_rv.rv_err, fmt="k.")
for idx in plot_inds:
    ax.plot(t_fine_rv, idata.posterior["rv_save"].values.reshape(-1, t_fine_rv.size)[idx], "C1", alpha=0.1)
ax.set_ylabel("RV [m/s]")
ax.set_xlabel("MJD")
plt.show()

This does not look too far from the results in the paper, considering that we did not include planet c, the GRAVITY data or the SPHERE/GPI corrections.